In [2]:
import json
import logging
import sys
import time
import subprocess
import threading
import random
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, Any

import pandas as pd
import re
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor, as_completed

import langchain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import OllamaLLM

# =========================
# CONFIG
# =========================
MODEL_NAME = "llama2-uncensored:7b"  # change to your Ollama model
TEMPERATURE = 0.2
REPEATS = 1000         # how many times to cycle the 5 d-factor buckets
MAX_WORKERS = 1        # concurrency level
INVOKE_TIMEOUT = 240   # per-call timeout seconds
MAX_RETRIES = 2        # additional retry attempts
BASE_BACKOFF = 1.0     # backoff base seconds
RESTART_THROTTLE_SEC = 30

# File paths (responder-specific)
PROMPT_PATH = "_responder.txt"                        # your fixed 32/8 prompt
PERSONALITY_CSV = "d_traits.csv"                      # provides {d_description}
OUTPUT_CSV = f"_data_responder_{MODEL_NAME.replace(':', '_').replace('-', '_')}_temp{TEMPERATURE}_{REPEATS}.csv"
MALFORMED_RESPONSES_LOG = "malformed_responses_samples_responder.jsonl"
RUN_LOG_FILE = "run_responder.log"

# =========================
# LOGGING
# =========================
# console + file logging with timestamps; UTF-8 file
console_handler = logging.StreamHandler(sys.stdout)
file_handler = logging.FileHandler(RUN_LOG_FILE, encoding="utf-8")
fmt = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
console_handler.setFormatter(fmt)
file_handler.setFormatter(fmt)

base_logger = logging.getLogger("sim")
base_logger.setLevel(logging.INFO)
# avoid duplicate handlers if re-run in notebooks
if not base_logger.handlers:
    base_logger.addHandler(console_handler)
    base_logger.addHandler(file_handler)

langchain.verbose = True

class ContextAdapter(logging.LoggerAdapter):
    def process(self, msg, kwargs):
        extra = self.extra.copy()
        parts = [f"{k}={v}" for k, v in extra.items() if v is not None]
        prefix = "[" + " ".join(parts) + "] " if parts else ""
        return prefix + msg, kwargs

def make_logger(agent_id=None, retry=None):
    return ContextAdapter(base_logger, {"agent_id": agent_id, "retry": retry})

# =========================
# OLLAMA PROCESS MGMT (Windows)
# =========================
ollama_restarting_event = threading.Event()
restart_lock = threading.Lock()
_last_restart_time = 0.0

def is_ollama_running() -> bool:
    try:
        output = subprocess.check_output(["tasklist", "/FI", "IMAGENAME eq ollama.exe"], stderr=subprocess.STDOUT)
        return b"ollama.exe" in output
    except Exception as e:
        base_logger.error(f"Error checking Ollama process: {e}")
        return False

def restart_ollama():
    global _last_restart_time
    with restart_lock:
        now = time.time()
        if now - _last_restart_time < RESTART_THROTTLE_SEC:
            base_logger.info("Restart throttled; skipping.")
            return
        _last_restart_time = now

        base_logger.info("Restarting Ollama...")
        try:
            subprocess.run(["taskkill", "/F", "/IM", "ollama.exe"], stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                           check=True)
        except Exception as e:
            base_logger.warning(f"Failed to kill Ollama cleanly: {e}")

        wait_time = 0
        while is_ollama_running() and wait_time < 20:
            base_logger.info("Waiting for Ollama to shutdown...")
            time.sleep(2)
            wait_time += 2
        if is_ollama_running():
            base_logger.error("Ollama did not terminate in time.")
        else:
            base_logger.info("Ollama terminated.")

        try:
            subprocess.Popen(["ollama", "start"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        except Exception as e:
            base_logger.error(f"Error starting Ollama: {e}")

        wait_time = 0
        while not is_ollama_running() and wait_time < 20:
            base_logger.info("Waiting for Ollama to start...")
            time.sleep(2)
            wait_time += 2
        if not is_ollama_running():
            base_logger.error("Ollama did not start in time.")
        else:
            base_logger.info("Ollama restarted successfully.")

def wait_for_ollama():
    while ollama_restarting_event.is_set():
        time.sleep(0.5)

# =========================
# DATA MODELS
# =========================
@dataclass
class Agent:
    ID: int
    d: str
    Token: str
    d_description: str

# =========================
# HELPERS
# =========================
def load_csv_validated(path: str, required_cols: list):
    df = pd.read_csv(path)
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"CSV {path} missing required columns: {missing}")
    return df

def load_prompt(file_path: str) -> str:
    return Path(file_path).read_text(encoding="utf-8")

def normalize_decision(token: str) -> Optional[str]:
    if not token:
        return None
    t = token.strip().upper()
    accept_set = {"ACCEPT", "A", "YES", "Y", "TAKE", "ACCEPTED", "APPROVE", "APPROVAL"}
    reject_set = {"REJECT", "B", "NO", "N", "DECLINE", "REJECTED", "REFUSE", "REFUSAL", "DENY", "DENIED"}
    if t in accept_set:
        return "ACCEPT"
    if t in reject_set:
        return "REJECT"
    return None

def save_malformed_sample(agent_id, raw_response, parsed_decision, parsed_justification):
    sample = {
        "timestamp": time.time(),
        "agent_id": agent_id,
        "raw_response": raw_response,
        "parsed_decision": parsed_decision,
        "parsed_justification": parsed_justification
    }
    with open(MALFORMED_RESPONSES_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

def parse_response(response: str) -> Tuple[Optional[str], Optional[str]]:
    try:
        stripped = response.strip()

        # JSON mode
        if stripped.startswith("{"):
            obj = json.loads(stripped)
            decision = normalize_decision(obj.get("decision", ""))
            justification = obj.get("justification", "").strip()
            if decision and justification:
                return decision, justification

        # "Decision: Accept/Reject" + "Justification: ..."
        m_dec = re.search(r'Decision:\s*(ACCEPT|REJECT)', response, re.IGNORECASE)
        m_jus = re.search(r'Justification:\s*(.+)', response, re.DOTALL | re.IGNORECASE)
        if m_dec and m_jus:
            decision = normalize_decision(m_dec.group(1))
            justification = m_jus.group(1).strip()
            if decision and justification:
                return decision, justification

        # Fallback to A/B
        m_dec_ab = re.search(r'Decision:\s*([ABab])', response)
        if m_dec_ab and m_jus:
            decision = normalize_decision(m_dec_ab.group(1))
            justification = m_jus.group(1).strip()
            if decision and justification:
                return decision, justification
    except Exception:
        pass
    return None, None

# =========================
# LLM CHAIN
# =========================
llm = OllamaLLM(model=MODEL_NAME, temperature=TEMPERATURE, num_predict=4096)
prompt_text = load_prompt(PROMPT_PATH)
template = ChatPromptTemplate.from_template(prompt_text)
responder_chain = template | llm | StrOutputParser()

def invoke_with_retries(payload: Dict[str, Any], context: Dict[str, Any]) -> Tuple[str, Optional[str], Optional[str]]:
    last_exc = None
    for attempt in range(1, MAX_RETRIES + 2):
        log = make_logger(agent_id=context.get("agent_id"), retry=attempt)
        log.info("Invoke attempt starting.")
        try:
            wait_for_ollama()
            with ThreadPoolExecutor(max_workers=1) as exec_single:
                future = exec_single.submit(lambda: responder_chain.invoke(payload))
                raw = future.result(timeout=INVOKE_TIMEOUT)
            decision, justification = parse_response(raw)
            if decision and justification:
                log.info(f"Invoke attempt success. Parsed decision={decision}.")
                return raw, decision, justification
            log.warning("Malformed output; saving sample.")
            save_malformed_sample(context.get("agent_id"), raw, decision, justification)
            if attempt > MAX_RETRIES:
                return raw, decision, justification
            backoff = BASE_BACKOFF * (2 ** (attempt - 1)) + random.uniform(0, 0.5)
            log.info(f"Retry backoff {backoff:.2f}s.")
            time.sleep(backoff)
        except concurrent.futures.TimeoutError:
            log.error("Timeout on invoke.")
            last_exc = "timeout"
            if attempt == 1:
                ollama_restarting_event.set()
                restart_ollama()
                ollama_restarting_event.clear()
            backoff = BASE_BACKOFF * (2 ** (attempt - 1)) + random.uniform(0, 0.5)
            log.info(f"Retry backoff {backoff:.2f}s after timeout.")
            time.sleep(backoff)
        except Exception as e:
            log.error(f"Unexpected error: {e}")
            last_exc = e
            if attempt == 1:
                ollama_restarting_event.set()
                restart_ollama()
                ollama_restarting_event.clear()
            backoff = BASE_BACKOFF * (2 ** (attempt - 1)) + random.uniform(0, 0.5)
            log.info(f"Retry backoff {backoff:.2f}s after exception.")
            time.sleep(backoff)
    return f"Error after retries: {last_exc}", None, None

# =========================
# PIPELINE
# =========================
def process_responder(agent: Agent) -> Dict[str, Any]:
    """
    One decision per agent. Prompt only needs {d_value} and {d_description}.
    """
    log = make_logger(agent_id=agent.ID)
    start = time.time()
    log.info(f"AGENT_START token={agent.Token} d={agent.d}")

    context = {"agent_id": agent.ID}
    payload = {
        "d_value": agent.d,
        "d_description": agent.d_description,
    }
    raw_resp, decision, justification = invoke_with_retries(payload, context)
    if decision is None or justification is None:
        log.warning(f"Final parse failure. decision={decision} justification={justification}")

    dur = time.time() - start
    log.info(f"AGENT_END token={agent.Token} d={agent.d} duration_sec={dur:.2f} decision={decision}")

    return {
        "agent_id": agent.ID,
        "token": agent.Token,
        "d": agent.d,
        "d_description": agent.d_description,
        "q1_full_response": raw_resp,
        "q1_decision": decision,
        "q1_text": justification,
    }

def simulate_responses_responder(agents_df: pd.DataFrame) -> pd.DataFrame:
    agents = [Agent(ID=row["ID"], d=row["d"], Token=row["Token"], d_description=row["d_description"])
              for _, row in agents_df.iterrows()]
    results = []
    total = len(agents)
    base_logger.info(f"Submitting {total} agents with MAX_WORKERS={MAX_WORKERS}...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_map = {}
        for ag in agents:
            base_logger.info(f"SUBMIT agent_id={ag.ID} token={ag.Token} d={ag.d}")
            fut = executor.submit(process_responder, ag)
            future_map[fut] = ag

        for i, future in enumerate(as_completed(future_map), 1):
            agent = future_map[future]
            try:
                res = future.result()
                results.append(res)
                base_logger.info(f"COMPLETE agent_id={agent.ID} token={agent.Token} d={agent.d} ({i}/{total})")
            except Exception as exc:
                make_logger(agent_id=agent.ID).error(f"Agent crashed: {exc}")
            if i % max(1, total // 4) == 0 or i == total:
                base_logger.info(f"Progress {i}/{total} agents ({(i / total) * 100:.1f}%)")
    return pd.DataFrame(results)

# =========================
# MAIN
# =========================
if __name__ == "__main__":
    # Load personality descriptions (expects columns: trait, intensity, text_description)
    personality_df = load_csv_validated(PERSONALITY_CSV, required_cols=['trait', 'intensity', 'text_description'])

    # Build agents with repeats (5 buckets cycling)
    base_traits = ["Low", "Low-Moderate", "Moderate", "Moderate-High", "High"]
    trait_list = base_traits * REPEATS
    agents_df = pd.DataFrame(trait_list, columns=['d'])
    agents_df.insert(0, 'ID', range(1, len(agents_df) + 1))
    agents_df["Token"] = ["D" + str((i % 5) + 1) for i in range(len(agents_df))]

    # Merge personality descriptions defensively
    personality_df['merge_key'] = personality_df['trait'].str.strip() + personality_df['intensity'].astype(str)
    personality_df.set_index('merge_key', inplace=True)
    agents_df['d_key'] = "d" + agents_df['d'].astype(str)  # expects trait='d' and intensity exactly matching strings above
    agents_df = pd.merge(agents_df,
                         personality_df[['text_description']],
                         left_on='d_key',
                         right_index=True,
                         how='left')
    agents_df.rename(columns={'text_description': 'd_description'}, inplace=True)
    if agents_df['d_description'].isnull().any():
        missing = agents_df[agents_df['d_description'].isnull()]
        base_logger.warning(f"{len(missing)} agents missing personality descriptions. IDs: {missing['ID'].tolist()}")
        # Fill with empty string to avoid None in prompt
        agents_df['d_description'] = agents_df['d_description'].fillna("")

    # Run the simulation (single fixed 32/8 decision per agent)
    simulation_results = simulate_responses_responder(agents_df)
    simulation_results.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    base_logger.info(f"Wrote results to {OUTPUT_CSV}")
    print(simulation_results.head())

2025-08-24 17:28:35,808 - INFO - Submitting 5000 agents with MAX_WORKERS=1...
2025-08-24 17:28:35,808 - INFO - SUBMIT agent_id=1 token=D1 d=Low
2025-08-24 17:28:35,809 - INFO - [agent_id=1] AGENT_START token=D1 d=Low
2025-08-24 17:28:35,809 - INFO - SUBMIT agent_id=2 token=D2 d=Low-Moderate
2025-08-24 17:28:35,809 - INFO - [agent_id=1 retry=1] Invoke attempt starting.
2025-08-24 17:28:35,811 - INFO - SUBMIT agent_id=3 token=D3 d=Moderate
2025-08-24 17:28:35,812 - INFO - SUBMIT agent_id=4 token=D4 d=Moderate-High
2025-08-24 17:28:35,812 - INFO - SUBMIT agent_id=5 token=D5 d=High
2025-08-24 17:28:35,812 - INFO - SUBMIT agent_id=6 token=D1 d=Low
2025-08-24 17:28:35,814 - INFO - SUBMIT agent_id=7 token=D2 d=Low-Moderate
2025-08-24 17:28:35,815 - INFO - SUBMIT agent_id=8 token=D3 d=Moderate
2025-08-24 17:28:35,816 - INFO - SUBMIT agent_id=9 token=D4 d=Moderate-High
2025-08-24 17:28:35,816 - INFO - SUBMIT agent_id=10 token=D5 d=High
2025-08-24 17:28:35,816 - INFO - SUBMIT agent_id=11 token=D

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



2025-08-24 17:28:39,004 - INFO - [agent_id=2] AGENT_END token=D2 d=Low-Moderate duration_sec=0.96 decision=REJECT
2025-08-24 17:28:39,004 - INFO - [agent_id=3] AGENT_START token=D3 d=Moderate
2025-08-24 17:28:39,005 - INFO - [agent_id=3 retry=1] Invoke attempt starting.
2025-08-24 17:28:39,005 - INFO - COMPLETE agent_id=2 token=D2 d=Low-Moderate (2/5000)
2025-08-24 17:28:40,107 - INFO - [agent_id=3 retry=1] Invoke attempt success. Parsed decision=REJECT.
2025-08-24 17:28:40,107 - INFO - [agent_id=3] AGENT_END token=D3 d=Moderate duration_sec=1.10 decision=REJECT
2025-08-24 17:28:40,107 - INFO - [agent_id=4] AGENT_START token=D4 d=Moderate-High
2025-08-24 17:28:40,107 - INFO - COMPLETE agent_id=3 token=D3 d=Moderate (3/5000)
2025-08-24 17:28:40,107 - INFO - [agent_id=4 retry=1] Invoke attempt starting.
2025-08-24 17:28:41,782 - INFO - [agent_id=4 retry=1] Invoke attempt success. Parsed decision=REJECT.
2025-08-24 17:28:41,782 - INFO - [agent_id=4] AGENT_END token=D4 d=Moderate-High dura